# Phase 4 — Content-Based Recommender
Two approaches: TF-IDF on title+genres, and dense Sentence-Transformer embeddings. User profile = weighted mean of item vectors (weights = rating − 2.5).

In [ ]:
import sys, time
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from src.data import load_movies
from src.content import TFIDFRecommender, SentenceTransformerRecommender

In [ ]:
train  = pd.read_csv('../data/train.csv')
val    = pd.read_csv('../data/val.csv')
movies = load_movies()
print(f'movies: {movies.shape}  train: {train.shape}  val: {val.shape}')
movies.head(3)

## 1. TF-IDF Recommender

In [ ]:
t0    = time.time()
tfidf = TFIDFRecommender(max_features=500).fit(movies, train)
print(f'TF-IDF fit in {time.time()-t0:.2f}s')
print(f'Item matrix shape: {tfidf.item_vectors.shape}')

In [ ]:
USER_ID = 1
recs = tfidf.recommend(user_id=USER_ID, train_df=train, n=10)
rec_df = pd.DataFrame(recs, columns=['movieId', 'score'])
rec_df = rec_df.merge(movies[['movieId', 'title', 'genres']], on='movieId')
print(f'TF-IDF recs for user {USER_ID}:')
rec_df

In [ ]:
def precision_at_k(model, train_df, val_df, k=10, n_users=200, use_train=True):
    val_items = val_df.groupby('userId')['movieId'].apply(set).to_dict()
    sample_users = list(val_items.keys())[:n_users]
    hits = []
    for uid in sample_users:
        if use_train:
            recs = model.recommend(uid, train_df=train_df, n=k)
        else:
            recs = model.recommend(uid, n=k)
        rec_ids = {r[0] for r in recs}
        hits.append(len(rec_ids & val_items[uid]) / k)
    return np.mean(hits)

p_tfidf = precision_at_k(tfidf, train, val, k=10)
print(f'TF-IDF  Precision@10 (first 200 users): {p_tfidf:.4f}')

## 2. Sentence-Transformer Recommender

In [ ]:
t0 = time.time()
st = SentenceTransformerRecommender(model_name='all-MiniLM-L6-v2').fit(movies, train)
print(f'Sentence-Transformer fit in {time.time()-t0:.1f}s')
print(f'Embedding matrix shape: {st.item_embs.shape}')

In [ ]:
recs = st.recommend(user_id=USER_ID, train_df=train, n=10)
rec_df2 = pd.DataFrame(recs, columns=['movieId', 'score'])
rec_df2 = rec_df2.merge(movies[['movieId', 'title', 'genres']], on='movieId')
print(f'Sentence-Transformer recs for user {USER_ID}:')
rec_df2

In [ ]:
p_st = precision_at_k(st, train, val, k=10)
print(f'Sentence-Transformer  Precision@10 (first 200 users): {p_st:.4f}')

## 3. Summary

| Model | Precision@10 |
|---|---|
| User-User CF | 0.0015 |
| Matrix Factorization | 0.0070 |
| Sentence-Transformer | 0.0060 |
| TF-IDF Content | 0.0160 |
| Item-Item CF | 0.0855 |

TF-IDF beats Sentence-Transformer here — genre+title text is short and keyword-heavy, so exact term matching (TF-IDF) works better than semantic embeddings (ST). ST shines with richer free-text descriptions. Both content models are weaker than Item-Item CF because they ignore actual user interaction patterns — a user's rating history is a stronger signal than what the movie text says.